# DiCo-NLI · AugB v7 · Commit 6 DirEnt（Kaggle T4 x2）

训练脚本 `train_augb_v7.py`（v7.2，Claude：v6.1 加 Commit 6，其他组的训练路径不变；v7.1 按 Codex 的审查让两个并行 worker 不再写共享文件，并逐项核对 SMOKE 回执的数值；v7.2 让 DEC / DEC-H 的 SMOKE 一直取 batch 到 2 次实际更新为止，开头的 fp16 跳步最多 2 次、如实记录）。本次在只有原始英语 fit 的数据（和 B0 相同）上，编码器用 Commit 4 NLIinit 的同一个 NLI 检查点初始化，只改输出结构：模型给"x 能推出 y"打一个分 s(x, y)，两个顺序各打一次，四类概率由两个方向的蕴含组合出来（E = 两边都蕴含，F / B = 只有一边，O = 都不蕴含），所以原方向和对称化的预测按构造相同。

- `DEC`：新建的 1 维打分头（单独的随机数生成器初始化）；
- `DEC-H`：保留 NLI 检查点自己的三分类头，打分 = 蕴含对其余两类的 log-odds，整头一起训练；
- 两组都只用每行自己这个方向的 BCE（E / F 为正例），训练行、顺序和计划更新次数和 NLIinit 相同；预测时某个顺序 s > 0 才算蕴含（s 恰好为 0 算不蕴含）。

对照组是 Commit 4 的 NLIinit：SMOKE 里重跑一次 NLIinit 的两步，和 Commit 4 归档的 SMOKE 逐格比对，再核对 15 个 Commit 4 任务的身份（种子、折、训练行数、步数、数据哈希、模型、配置）。都通过就复用，正式队列 30 个 task；不通过就在同一队列里重跑 NLIinit，共 45 个。按 NLIinit 每个 task 约 4 分钟算，30 个约 2 GPU 小时，T4 x2 墙钟约 1 小时（45 个约 1.5 小时；加安装和 SMOKE 再加约 20 分钟）。

**设置**：Accelerator 选 **GPU T4 x2**；右侧 Add Input 挂上本次的训练包（含 `train_augb_v7.py`）和官方仓库数据集；**打开 Internet**（要下载 NLI 检查点，约 560 MB）。

**第一次**：`RUN_FORMAL = False`，交互会话 Run All，只跑预检和 SMOKE（NLIinit / DEC / DEC-H 各两步，加数值检查、DEC-H 初始值和 NLI 检查点的核对、零样本 DEC-H），把打印出来的结果包发给 Claude。三组都必须是 2 次实际更新；NLIinit 必须 0 次跳步（要和 Commit 4 逐格比对），DEC / DEC-H 开头允许最多 2 次 fp16 跳步（GradScaler 从 65536 往下调，正式训练里也会有）；不满足时 SMOKE 或正式队列会在第一个 task 前停下。

**正式**：`RUN_FORMAL = True`，Save Version → Save & Run All (Commit)。跑完在版本页 Output 下载 `augb_v7_c6_results.zip`。中断后把上一个版本的 Output 用 Add Input 挂上，再 Commit 一次，已完成的 task 自动跳过。续跑的前提是 torch / cuda / transformers 和上一次相同：版本不同时，正式队列会在第一个 task 前停下并列出旧 task，这时不要挂旧 Output，从头跑。


In [ ]:
RUN_FORMAL = False
COMMIT = "6"
import os, sys, glob, pathlib, subprocess, zipfile, shutil, json, time
os.environ["TOKENIZERS_PARALLELISM"] = "false"
ROOT = pathlib.Path('/kaggle/working/augb_v7')
OUT = pathlib.Path(f'/kaggle/working/augb_v7_c{COMMIT}_runs')
RESULT_ZIP = pathlib.Path(f'/kaggle/working/augb_v7_c{COMMIT}_results.zip')

def pack(note=''):
    if not OUT.exists():
        print('还没有结果目录', OUT); return
    tmp = RESULT_ZIP.with_suffix('.tmp'); n = 0
    with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUT.rglob('*')):
            if p.is_file() and p.suffix != '.pt':
                z.write(p, p.relative_to(OUT.parent)); n += 1
    tmp.replace(RESULT_ZIP)
    done = len(list(OUT.glob('tasks/**/done.json')))
    print(f'[结果包] {RESULT_ZIP}  {RESULT_ZIP.stat().st_size/1e6:.2f} MB，{n} 个文件；正式 task 完成 {done} {note}')
    try:
        from IPython.display import FileLink, display
        display(FileLink(str(RESULT_ZIP.relative_to('/kaggle/working'))))
    except Exception:
        pass
gpus = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip().splitlines()
print('RUN_FORMAL =', RUN_FORMAL, '| COMMIT =', COMMIT, '| GPU:', gpus)


In [ ]:
# transformers 钉在 Commit 4 用的 4.57.6；torch 用镜像自带的
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers==4.57.6','sentencepiece>=0.2','protobuf>=4.25','scikit-learn>=1.3'], check=True)
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__)
if torch.__version__ != '2.11.0+cu128':
    print('提醒：Commit 4 用的是 torch 2.11.0+cu128；版本不同时 NLIinit 复用检查不会通过，NLIinit 会在本次重跑（45 个 task）')


In [ ]:
# 训练包：Kaggle 会把上传的 zip 解压成文件夹；找同时有 train_augb_v7.py、data/gate.json、data_v7/c4_smoke_NLIinit 的那个
hits = [p.parent for p in pathlib.Path('/kaggle/input').glob('**/train_augb_v7.py')
        if (p.parent/'data/gate.json').is_file() and (p.parent/'data_v3/mix_lists.json').is_file()
        and (p.parent/'data_v7/c4_smoke_NLIinit/predictions.csv').is_file() and (p.parent/'data_v7/c4_nliinit_tasks').is_dir()
        and not (p.parent.parent/OUT.name).exists()]
if len(hits) != 1:
    raise RuntimeError('需要恰好一个训练包（含 train_augb_v7.py），找到：' + ', '.join(map(str, hits)))
if ROOT.exists(): shutil.rmtree(ROOT)
shutil.copytree(hits[0], ROOT)
print('Training package:', hits[0], '→', ROOT)


In [ ]:
def find_official():
    c = list(pathlib.Path('/kaggle/input').glob('**/final_data/train/dico_nli_train_track1_reference.csv'))
    if c: return c[0].parents[2]
    clone = pathlib.Path('/kaggle/working/SemEval-2027-Task-2-DiCo-NLI')
    if not clone.exists():
        subprocess.run(['git','clone','--depth','1','https://github.com/ilopezgazpio/SemEval-2027-Task-2-DiCo-NLI.git',str(clone)], check=True)
    return clone
OFFICIAL = find_official(); print('Official repo:', OFFICIAL)
BASE = [sys.executable, str(ROOT/'train_augb_v7.py'), '--bundle', str(ROOT), '--official', str(OFFICIAL), '--out', str(OUT), '--commit', COMMIT,
        '--model-revision', 'a0484667b22365f84929a935b5e50a51f71f159d']
subprocess.run(BASE + ['--stage', 'preflight'], check=True)


In [ ]:
# SMOKE（GPU 0）：NLIinit / DEC / DEC-H 各两步，加数值检查、DEC-H 初始值核对、零样本 DEC-H 和对照组身份核对；跑完马上打包
env = dict(os.environ, CUDA_VISIBLE_DEVICES='0')
try:
    subprocess.run(BASE + ['--stage', 'smoke'], check=True, env=env)
finally:
    for p in sorted((OUT/'smoke').glob('*/smoke.json')):
        s = json.loads(p.read_text())
        print(f"[SMOKE] {s['group']:8s} 更新 {s['updates']}  跳步 {s['skips']}  WF1 {s['validation_wf1']:.4f}  用时 {s['elapsed_sec']:.0f}s  {s['stable_environment']}")
        ar = s.get('arm_report') or {}
        nli, di = ar.get('nli_init'), ar.get('dec_init')
        if nli:
            print(f"        NLI 载入 {nli['tensors_loaded']} 个张量、{nli['parameters_loaded']:,} 个参数；保留 B0 的 {nli['kept_from_b0']}；前向配置差别 {nli['forward_config_differences_kept_b0']}")
        if di:
            print(f"        方向头 {di}")
    for name in ('nli_reuse.json', 'dec_checks.json'):
        p = OUT/name
        if p.exists():
            r = json.loads(p.read_text())
            if name == 'nli_reuse.json':
                print('[NLIinit 复用]', r['status'], r['differences'])
            else:
                print('[对照组身份]', r.get('control_identity'), '| DEC-H 初值', (r.get('init') or {}).get('DEC-H'))
    pack('（SMOKE）')


In [ ]:
# 续跑：挂了上一次的 Output（文件夹或 zip）就恢复已完成的 task
dirs = sorted({p.parent.parent for p in pathlib.Path('/kaggle/input').glob(f'**/{OUT.name}/tasks/*')})
zips = [] if dirs else sorted(glob.glob(f'/kaggle/input/**/{RESULT_ZIP.name}', recursive=True))
if len(dirs) > 1 or len(zips) > 1: raise RuntimeError('找到多份旧结果，请只挂一份')
if dirs:
    shutil.copytree(dirs[0]/'tasks', OUT/'tasks', dirs_exist_ok=True, ignore=shutil.ignore_patterns('*.pt')); print('从文件夹恢复：', dirs[0])
elif zips:
    with zipfile.ZipFile(zips[0]) as z:
        names = [m for m in z.namelist() if pathlib.PurePosixPath(m).parts[:2] == (OUT.name, 'tasks')]
        for m in names:
            pp = pathlib.PurePosixPath(m)
            if pp.is_absolute() or '..' in pp.parts: raise ValueError(m)
        z.extractall('/kaggle/working', members=names)
    print('从 zip 恢复：', zips[0])
print('已有 done.json：', len(list(OUT.glob('tasks/**/done.json'))))


In [ ]:
# 正式：每张 GPU 一个 worker 进程，task 按估计用时分给两边；每 10 分钟打印一次进度；无论成败最后都打包
if RUN_FORMAL:
    n = max(1, len(gpus))
    (OUT/'logs').mkdir(parents=True, exist_ok=True)
    procs = []
    for i in range(n):
        log = open(OUT/'logs'/f'worker{i}.log', 'a')
        procs.append(subprocess.Popen(BASE + ['--stage', 'full', '--worker', str(i), '--workers', str(n)],
                                      stdout=log, stderr=subprocess.STDOUT, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(i))))
    t0 = time.time()
    try:
        while any(p.poll() is None for p in procs):
            time.sleep(600)
            print(f'[{(time.time()-t0)/3600:.1f} h] done.json：', len(list(OUT.glob('tasks/**/done.json'))), flush=True)
            for i in range(n):
                tail = (OUT/'logs'/f'worker{i}.log').read_text(errors='replace').strip().splitlines()[-2:]
                print(f'  worker{i}:', ' | '.join(tail)[-300:], flush=True)
    finally:
        print('worker 退出码：', [p.returncode for p in procs])
        pack('（正式）')
        shutil.rmtree(ROOT, ignore_errors=True)
    if any(p.returncode for p in procs):
        raise RuntimeError('有 worker 出错，见 logs/worker*.log（已打包）')
else:
    print('只跑了 SMOKE。把 RUN_FORMAL 改成 True，再 Save Version → Save & Run All (Commit)。')
